In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime, timedelta
from tqdm import tqdm
import re
import locale
import urllib3
import emoji

# Suppress the InsecureRequestWarning specifically
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [2]:

def get_html(url = 'https://www.sindipetrorn.org.br/noticias/'):
    payload = {}
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36",
        "Accept-Language": "pt-BR,pt;q=0.9,en;q=0.8",
        "Referer": "https://google.com"
    }

    response = requests.request("GET", url, headers=headers, data=payload, verify=False)
    html_content = response.text

    return html_content

In [16]:
def get_links_and_dates(html_content):
    soup = BeautifulSoup(html_content, 'html.parser')
    articles = soup.find_all('div', class_='post-card-wrapper mb-4')

    locale.setlocale(locale.LC_TIME, "pt_BR.utf8") 

    news_links = []

    for i in range(len(articles)):
        a = articles[i].find('a')
        link = a.get('href')
        span = articles[i].find('time')
        date = span.text.strip()
        try:
            date = datetime.strptime(date, "%d de %B de %Y")
                
            link_date = [link, date]
            news_links.append(link_date)
        except ValueError as e:
            continue

    articles = soup.find_all('div', class_='gallery-card-wrapper mb-5')
    
    for i in range(len(articles)):
        a = articles[i].find('a')
        link = a.get('href')
        span = articles[i].find('time')
        date = span.text.strip()
        try:
            date = datetime.strptime(date, "%d de %B de %Y")
                
            link_date = [link, date]
            news_links.append(link_date)
        except ValueError as e:
            continue


    return news_links

In [24]:
def get_validated_links(news_links, min_date = datetime(2025,6,1)):
    validated_links = []
    next_page = True
    for link, date in news_links:
        if date < min_date:
            next_page = False
            break
        else:
            validated_links.append([link, date])

    return validated_links, next_page

In [25]:
def get_next_page(fnp_url = 'https://www.sindipetrorn.org.br/noticias/', next_page_number = 1):
    validated_news_links = []
    url = fnp_url + 'page/' + str(next_page_number) + '/'
    html_content = get_html(url)
    news_links = get_links_and_dates(html_content)
    validated_links, next_page = get_validated_links(news_links)

    for validated_link in validated_links:
        validated_news_links.append(validated_link)

    if next_page:
        validated_links = get_next_page(fnp_url, next_page_number + 1)
        
        for validated_link in validated_links:
            validated_news_links.append(validated_link)

    return validated_news_links

In [28]:
def sanitize_paragraphs(paragraphs:list, min_paragraph_len = 500, concat_trigger_size = 1500):
    '''
    sanitiza os parágrafos, realizando replace de partes de strings e concatenando parágrafos
    
    paragraphs: lista de parágrafos
    min_paragraph_len: define quais parágrafos devem ser concatenados
    concat_trigger_size: caso o tamanho da concatenação de textos esteja acima desse valor, 
                            escreve como parágrafo e reseta a variável que armazena as 
                            strings a serem concatenadas
    '''
    def append_paragraphs(current_new_paragraph):
        new_paragraphs.append(current_new_paragraph.strip())
    
    paragraphs = [emoji.demojize(paragraph) for paragraph in [paragraph\
                                                              .replace('\xa0',' ')\
                                                              .replace('\n',' ')\
                                                              .replace('\t',' ')\
                                                              .replace('[email-protected]', '')\
                                                              .strip() \
                                                              for paragraph in paragraphs] \
                  if len(paragraph)>0] #removendo emoji, tratanto texto e realizando strip para então construir lista de parágrafos que tenham len > 0
    paragraphs_mask = [len(paragraph) < min_paragraph_len for paragraph in paragraphs] # true são os abaixo do min_paragraph_len, precisarao ser tratados
    if any(paragraphs_mask): #caso algum elemento precise ser tratado, trigga processo
        new_paragraphs = []
        current_new_paragraph = ''
        for i in range(len(paragraphs)):
            if len(current_new_paragraph) >= concat_trigger_size:
                append_paragraphs(current_new_paragraph)
                current_new_paragraph = ''
            if paragraphs_mask[i]: #caso seja menor que o min_paragraph_len
                current_new_paragraph = current_new_paragraph + ' ' + paragraphs[i]
                if i == len(paragraphs)-1: #caso seja o ultimo elemento
                    append_paragraphs(current_new_paragraph)
            else:
                current_new_paragraph = current_new_paragraph + '' + paragraphs[i]
                append_paragraphs(current_new_paragraph)
                current_new_paragraph = ''
                    
    return new_paragraphs

In [29]:
def get_content_news(url):
    html_content = get_html(url)
    soup = BeautifulSoup(html_content, 'html.parser')
    
    title = soup.find('h1', class_='entry-title post-title').text
    paragraphs = soup.find('div', class_='col-md-8 mb-4')

    paragraphs = paragraphs.text.split('\n')
    paragraphs = sanitize_paragraphs(paragraphs)

    return title, paragraphs

In [ ]:
def main():
    next_page_number = 1
    validated_news_links = []
    url_default = 'https://www.sindipetrorn.org.br/noticias/'
    url = url_default + 'page/' + str(next_page_number) + '/'
    html_content = get_html(url)
    news_links = get_links_and_dates(html_content)
    validated_links, next_page = get_validated_links(news_links)

    for validated_link in validated_links:
        validated_news_links.append(validated_link)

    if next_page:
        validated_links = get_next_page(url_default, next_page_number + 1)
        
        for validated_link in validated_links:
            validated_news_links.append(validated_link)

    # result = []
    # for url, date in tqdm(validated_news_links):
    #     title, paragraphs = get_content_news(url)
    #     num_paragraph = 1
    #     for paragraph in paragraphs:
    #         result.append(
    #             {
    #                 'sindicato': 'RS',
    #                 'url' : url,
    #                 'titulo' : title,
    #                 'data': date,
    #                 'paragrafo' : paragraph,
    #                 'num_paragrafo' : num_paragraph
    #             }
    #         )
    #         num_paragraph += 1

    return result

In [27]:
result = main()
print(len(result))
result

30


[['https://www.sindipetrorn.org.br/noticia/nota-de-solidariedade-a-vereadora-brisa-bracchi/',
  datetime.datetime(2025, 8, 21, 0, 0)],
 ['https://www.sindipetrorn.org.br/noticia/petroleiras-do-sindipetro-rn-marcam-presenca-e-protagonismo-em-espacos-nacionais-de-luta/',
  datetime.datetime(2025, 8, 14, 0, 0)],
 ['https://www.sindipetrorn.org.br/noticia/sindipetro-rn-participa-de-ato-nacional-pelo-fim-dos-peds-na-petros/',
  datetime.datetime(2025, 8, 13, 0, 0)],
 ['https://www.sindipetrorn.org.br/noticia/sindipetro-rn-sauda-retorno-da-petrobras-a-distribuicao-de-glp/',
  datetime.datetime(2025, 8, 12, 0, 0)],
 ['https://www.sindipetrorn.org.br/noticia/sindipetro-rn-leva-propostas-estrategicas-ao-6o-congresso-nacional-da-ctb/',
  datetime.datetime(2025, 8, 12, 0, 0)],
 ['https://www.sindipetrorn.org.br/noticia/ato-em-natal-cobra-pagamento-de-dividas-da-petrobras-com-a-petros/',
  datetime.datetime(2025, 8, 12, 0, 0)],
 ['https://www.sindipetrorn.org.br/noticia/sindipetro-rn-marca-presenc